In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

def preprocessing(data, typ, scaler=None, imputer=None):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13"]
    
    # ==================== CATEGORY AGGREGATIONS ====================
    # Basic sums
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1)
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1)
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1)
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1)
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1)
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1)
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1)
    
    # Statistical aggregations (mean, std, max, min)
    data['M*_mean'] = data[[f'M{i}' for i in range(1, 19)]].mean(axis=1)
    data['M*_std'] = data[[f'M{i}' for i in range(1, 19)]].std(axis=1)
    data['M*_max'] = data[[f'M{i}' for i in range(1, 19)]].max(axis=1)
    data['M*_min'] = data[[f'M{i}' for i in range(1, 19)]].min(axis=1)
    
    data['E*_mean'] = data[[f'E{i}' for i in range(1, 21)]].mean(axis=1)
    data['E*_std'] = data[[f'E{i}' for i in range(1, 21)]].std(axis=1)
    data['E*_max'] = data[[f'E{i}' for i in range(1, 21)]].max(axis=1)
    data['E*_min'] = data[[f'E{i}' for i in range(1, 21)]].min(axis=1)
    
    data['V*_std'] = data[[f'V{i}' for i in range(1, 14)]].std(axis=1)
    data['S*_std'] = data[[f'S{i}' for i in range(1, 13)]].std(axis=1)
    
    # Add category aggregates to features
    agg_features = ["M*", "E*", "I*", "P*", "V*", "S*", "D*"]
    main_features.extend(agg_features)
    main_features.extend(['M*_mean', 'M*_std', 'M*_max', 'M*_min', 
                         'E*_mean', 'E*_std', 'E*_max', 'E*_min',
                         'V*_std', 'S*_std'])
    
    # ==================== POLYNOMIAL FEATURES ====================
    # Squares and cubes for key categories
    for cat in ['M*', 'E*', 'V*', 'P*']:
        data[f'{cat}_sq'] = data[cat] ** 2
        data[f'{cat}_cube'] = data[cat] ** 3
        main_features.extend([f'{cat}_sq', f'{cat}_cube'])
    
    # Square roots (for non-negative emphasis)
    for cat in ['V*', 'S*']:
        data[f'{cat}_sqrt'] = np.sqrt(np.abs(data[cat]))
        main_features.append(f'{cat}_sqrt')
    
    # ==================== TWO-WAY INTERACTIONS ====================
    # M-E interactions
    data['ME_prod'] = data['M*'] * data['E*']
    data['ME_ratio'] = data['M*'] / (data['E*'] + 1e-8)
    data['ME_norm_diff'] = (data['M*'] - data['E*']) / (data['M*'] + data['E*'] + 1e-8)
    data['ME_harmonic'] = 2 * data['M*'] * data['E*'] / (data['M*'] + data['E*'] + 1e-8)
    main_features.extend(['ME_prod', 'ME_ratio', 'ME_norm_diff', 'ME_harmonic'])
    
    # M-I interactions
    data['MI_prod'] = data['M*'] * data['I*']
    data['MI_ratio'] = data['M*'] / (data['I*'] + 1e-8)
    data['MI_spread'] = data['M*'] - data['I*']
    data['MI_norm_diff'] = (data['M*'] - data['I*']) / (data['M*'] + data['I*'] + 1e-8)
    main_features.extend(['MI_prod', 'MI_ratio', 'MI_spread', 'MI_norm_diff'])
    
    # M-P interactions
    data['MP_prod'] = data['M*'] * data['P*']
    data['MP_ratio'] = data['M*'] / (data['P*'] + 1e-8)
    data['MP_weighted'] = data['M*'] * (1 + data['P*'])
    main_features.extend(['MP_prod', 'MP_ratio', 'MP_weighted'])
    
    # M-V interactions (volatility effects)
    data['MV_ratio'] = data['M*'] / (data['V*'] + 1e-8)
    data['MV_prod'] = data['M*'] * data['V*']
    data['MV_risk_adj'] = data['M*'] / (1 + data['V*'])
    data['MV_sharpe_like'] = data['M*'] / (data['V*_std'] + 1e-8)
    main_features.extend(['MV_ratio', 'MV_prod', 'MV_risk_adj', 'MV_sharpe_like'])
    
    # M-S interactions
    data['MS_prod'] = data['M*'] * data['S*']
    data['MS_weighted'] = data['M*'] * (1 + data['S*'])
    data['MS_momentum'] = data['M*'] * np.sign(data['S*'])
    main_features.extend(['MS_prod', 'MS_weighted', 'MS_momentum'])
    
    # M-D interactions
    data['MD_prod'] = data['M*'] * data['D*']
    data['MD_ratio'] = data['M*'] / (data['D*'] + 1e-8)
    main_features.extend(['MD_prod', 'MD_ratio'])
    
    # E-I interactions
    data['EI_diff'] = data['E*'] - data['I*']
    data['EI_ratio'] = data['E*'] / (data['I*'] + 1e-8)
    data['EI_prod'] = data['E*'] * data['I*']
    data['EI_norm_diff'] = (data['E*'] - data['I*']) / (data['E*'] + data['I*'] + 1e-8)
    main_features.extend(['EI_diff', 'EI_ratio', 'EI_prod', 'EI_norm_diff'])
    
    # E-P interactions
    data['EP_prod'] = data['E*'] * data['P*']
    data['EP_ratio'] = data['E*'] / (data['P*'] + 1e-8)
    data['EP_weighted'] = data['E*'] * (1 + data['P*'] / 100)
    main_features.extend(['EP_prod', 'EP_ratio', 'EP_weighted'])
    
    # E-V interactions
    data['EV_prod'] = data['E*'] * data['V*']
    data['EV_uncertainty'] = np.abs(data['E*']) * data['V*']
    data['EV_risk_adj'] = data['E*'] / (data['V*'] + 1e-8)
    main_features.extend(['EV_prod', 'EV_uncertainty', 'EV_risk_adj'])
    
    # E-S interactions
    data['ES_prod'] = data['E*'] * data['S*']
    data['ES_diff'] = data['E*'] - data['S*']
    data['ES_trend_strength'] = data['E*'] * np.abs(data['S*'])
    main_features.extend(['ES_prod', 'ES_diff', 'ES_trend_strength'])
    
    # I-P interactions
    data['IP_prod'] = data['I*'] * data['P*']
    data['IP_discount'] = data['P*'] / (1 + data['I*'] + 1e-8)
    data['IP_real_value'] = data['P*'] - data['I*']
    main_features.extend(['IP_prod', 'IP_discount', 'IP_real_value'])
    
    # I-V interactions
    data['IV_prod'] = data['I*'] * data['V*']
    data['IV_volatility_impact'] = data['V*'] * (1 + data['I*'] / 100)
    main_features.extend(['IV_prod', 'IV_volatility_impact'])
    
    # I-S interactions
    data['IS_prod'] = data['I*'] * data['S*']
    data['IS_real_sentiment'] = data['S*'] / (1 + data['I*'] + 1e-8)
    main_features.extend(['IS_prod', 'IS_real_sentiment'])
    
    # P-V interactions
    data['PV_prod'] = data['P*'] * data['V*']
    data['PV_stability'] = data['P*'] / (data['V*'] + 1e-8)
    data['PV_risk_reward'] = (data['P*'] / 100) / (data['V*'] + 1e-8)
    main_features.extend(['PV_prod', 'PV_stability', 'PV_risk_reward'])
    
    # P-S interactions
    data['PS_prod'] = data['P*'] * data['S*']
    data['PS_contrarian'] = -data['P*'] * data['S*']
    data['PS_momentum'] = data['P*'] * (1 + data['S*'] / 100)
    main_features.extend(['PS_prod', 'PS_contrarian', 'PS_momentum'])
    
    # V-S interactions
    data['VS_prod'] = data['V*'] * data['S*']
    data['VS_panic'] = data['V*'] * np.abs(data['S*'])
    data['VS_calm_bull'] = data['V*'] * data['S*'] * -1
    main_features.extend(['VS_prod', 'VS_panic', 'VS_calm_bull'])
    
    # V-D interactions
    data['VD_prod'] = data['V*'] * data['D*']
    data['VD_ratio'] = data['V*'] / (data['D*'] + 1e-8)
    main_features.extend(['VD_prod', 'VD_ratio'])
    
    # S-D interactions
    data['SD_prod'] = data['S*'] * data['D*']
    data['SD_divergence'] = data['S*'] - data['D*']
    main_features.extend(['SD_prod', 'SD_divergence'])
    
    # ==================== THREE-WAY INTERACTIONS ====================
    # M-E-V (momentum-earnings-volatility)
    data['MEV_prod'] = data['M*'] * data['E*'] * data['V*']
    data['MEV_risk_adj_return'] = (data['M*'] * data['E*']) / (data['V*'] + 1e-8)
    data['MEV_sharpe_extended'] = (data['M*'] + data['E*']) / (data['V*'] + 1e-8)
    main_features.extend(['MEV_prod', 'MEV_risk_adj_return', 'MEV_sharpe_extended'])
    
    # M-P-I (momentum-price-inflation)
    data['MPI_real_momentum'] = data['M*'] * data['P*'] / (1 + data['I*'] + 1e-8)
    data['MPI_prod'] = data['M*'] * data['P*'] * data['I*']
    main_features.extend(['MPI_real_momentum', 'MPI_prod'])
    
    # E-I-P (earnings-inflation-price)
    data['EIP_real_earnings'] = data['E*'] * data['P*'] / (1 + data['I*'] + 1e-8)
    data['EIP_value'] = (data['E*'] / (data['P*'] + 1e-8)) * (1 + data['I*'])
    main_features.extend(['EIP_real_earnings', 'EIP_value'])
    
    # V-S-D (volatility-sentiment-divergence)
    data['VSD_market_stress'] = data['V*'] * np.abs(data['S*'] - data['D*'])
    data['VSD_fear_index'] = data['V*'] * (1 - data['S*'] / 100) * data['D*']
    main_features.extend(['VSD_market_stress', 'VSD_fear_index'])
    
    # M-V-S (momentum-volatility-sentiment)
    data['MVS_quality'] = data['M*'] / (data['V*'] * (1 + np.abs(data['S*'])) + 1e-8)
    data['MVS_trend_confidence'] = data['M*'] * data['S*'] / (data['V*'] + 1e-8)
    main_features.extend(['MVS_quality', 'MVS_trend_confidence'])
    
    # ==================== MULTI-WAY SUMS (REGIMES) ====================
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*']
    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*']
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*']
    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*']
    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*']
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']
    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    main_features.extend(['M*_P*_V*', 'M*_E*_S*', 'M*_P*_I*_V*', 'V*_M*_E*_I*',
                         'V*_S*_D*', 'E*_I*_V*_D*', 'M*_V*_S*_D*', 'P*_V*_S*',
                         'P*_M*_D*', 'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'])
    
    # ==================== WEIGHTED MULTI-WAY COMBINATIONS ====================
    # Weighted by importance
    data['weighted_growth'] = 0.4*data['M*'] + 0.3*data['E*'] + 0.3*data['S*']
    data['weighted_risk'] = 0.5*data['V*'] + 0.3*data['D*'] + 0.2*np.abs(data['S*'])
    data['weighted_value'] = 0.5*data['P*'] + 0.3*data['E*'] - 0.2*data['I*']
    main_features.extend(['weighted_growth', 'weighted_risk', 'weighted_value'])
    
    # ==================== RATIO OF RATIOS ====================
    data['momentum_quality'] = (data['M*'] / (data['V*'] + 1e-8)) / (data['E*'] / (data['P*'] + 1e-8) + 1e-8)
    data['risk_return_ratio'] = (data['M*'] + data['E*']) / ((data['V*'] + data['D*']) + 1e-8)
    main_features.extend(['momentum_quality', 'risk_return_ratio'])
    
    # ==================== CROSS-CATEGORY STATISTICS ====================
    all_cats = ['M*', 'E*', 'I*', 'P*', 'V*', 'S*', 'D*']
    data['all_cat_mean'] = data[all_cats].mean(axis=1)
    data['all_cat_std'] = data[all_cats].std(axis=1)
    data['all_cat_max'] = data[all_cats].max(axis=1)
    data['all_cat_min'] = data[all_cats].min(axis=1)
    data['all_cat_range'] = data['all_cat_max'] - data['all_cat_min']
    main_features.extend(['all_cat_mean', 'all_cat_std', 'all_cat_max', 
                         'all_cat_min', 'all_cat_range'])
    
    # ==================== EXPONENTIAL TRANSFORMATIONS ====================
    data['M*_exp'] = np.exp(data['M*'] / (data['M*_std'] + 1))
    data['E*_exp'] = np.exp(data['E*'] / (data['E*_std'] + 1))
    main_features.extend(['M*_exp', 'E*_exp'])
    
    # ==================== DISTANCE METRICS ====================
    data['euclidean_MVS'] = np.sqrt(data['M*']**2 + data['V*']**2 + data['S*']**2)
    data['manhattan_MEIP'] = np.abs(data['M*']) + np.abs(data['E*']) + np.abs(data['I*']) + np.abs(data['P*'])
    main_features.extend(['euclidean_MVS', 'manhattan_MEIP'])
    
    # ==================== FINAL DATA PREPARATION ====================
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
        
        scaler = MinMaxScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
        
        return data, scaler, imputer
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
            
        data[feature_cols] = imputer.transform(data[feature_cols])
        data[feature_cols] = scaler.transform(data[feature_cols])
        
        return data

train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_processed, scaler, imputer = preprocessing(train, "train")
train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,all_cat_mean,all_cat_std,all_cat_max,all_cat_min,all_cat_range,M*_exp,E*_exp,euclidean_MVS,manhattan_MEIP,target
6969,0.041781,0.018889,0.080586,0.079422,0.205607,0.360656,0.048290,0.780305,0.733879,0.677652,...,0.397613,0.113082,0.069790,0.812129,0.099200,0.002481,0.016429,0.078050,0.273818,0.001145
6970,0.041500,0.018512,0.076923,0.075812,0.196262,0.344262,0.007713,0.780123,0.733868,0.677908,...,0.460149,0.094014,0.067672,0.856861,0.078203,0.003364,0.017261,0.077057,0.319792,0.004738
6971,0.041219,0.018134,0.073260,0.072202,0.186916,0.327869,0.007378,0.779941,0.733856,0.678165,...,0.479786,0.102995,0.069099,0.814135,0.097694,0.001894,0.017072,0.084986,0.354661,0.006016
6972,0.040938,0.017756,0.069597,0.068592,0.177570,0.311475,0.007042,0.776877,0.893087,0.678422,...,0.469166,0.121535,0.088038,0.751968,0.142022,0.000821,0.026458,0.091021,0.382904,0.001414
6973,0.040657,0.017378,0.065934,0.064982,0.168224,0.295082,0.006707,0.776701,0.892525,0.678680,...,0.453439,0.110810,0.084259,0.782952,0.125289,0.001260,0.022726,0.082745,0.351001,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.282092,0.210049,0.205128,0.202166,0.149533,0.262295,0.923541,0.627247,0.437709,0.479189,...,0.394433,0.067143,0.064575,0.849248,0.078503,0.005232,0.022310,0.067656,0.240100,0.002457
8986,0.281430,0.209671,0.201465,0.198556,0.140187,0.245902,0.923877,0.627242,0.437767,0.479013,...,0.428489,0.056655,0.054856,0.849248,0.069299,0.009703,0.014250,0.095586,0.258756,0.002312
8987,0.280770,0.209294,0.197802,0.194946,0.130841,0.229508,0.924212,0.627200,0.437777,0.478845,...,0.402175,0.040517,0.054331,0.886367,0.053043,0.008718,0.013691,0.080306,0.219471,0.002891
8988,0.280112,0.208916,0.194139,0.191336,0.121495,0.213115,0.924547,0.627159,0.437787,0.478676,...,0.383946,0.073419,0.058427,0.849248,0.072681,0.005398,0.016282,0.051529,0.254436,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {'boosting_type': 'gbdt', 
               'objective': 'regression_l1', 
               'metric': 'mape', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 152611150.3849
Training Model 2...
Cumulative Training MAPE: 3039671.8697
Training Model 3...
Cumulative Training MAPE: 5053001.0573
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 5053001.0573
MAE: 0.0015
RMSE: 0.0041


In [5]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)


def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()
        
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))